In [29]:
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.common.by import By

# 1. Configuracion de metadatos y conexion
NOMBRE_GRUPO = "G3_FinanzasyMercadoortega"
CATEGORIA_ACTUAL = "Finanzas" #Los alumnos pueden cambiar esto segun su asignacion

try:
    client = MongoClient('database',27017)
    db = client['proyecto_bigdata']
    coleccion = db ['datos_libros']
    print("CONEXION EXITOSA : Listo para insertar datos de", CATEGORIA_ACTUAL)
except Exception as e:
    print("ERROR DE CONEXION:",e)

 #2.Ejemplo de logica de captura (Simplificada para la clase)
 #Supongamos que ’lista_libros’ es el resultado de tu busqueda con Selenium
lista_libros=[
 {"titulo" : "It’s Only Himalaya", "precio": " 45.17"},
 {"titulo" : "Full Moon over Noahs Ark", "precio": " 49.43"}
 ]

for libro in lista_libros:
    try:
        #Enriquecimiento del diccionario
        registro={
            "item":libro["titulo"],
            "valor": float(libro["precio"].replace(' ','').replace(',','').strip()),
            "categoria": CATEGORIA_ACTUAL,
            "grupo": NOMBRE_GRUPO,
            "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        coleccion.insert_one(registro)
    except Exception as e:
        print("Errorprocesandolibro:",e)

print("PROCESO FINALIZADO : Datos enviados a MongoDB.")

CONEXION EXITOSA : Listo para insertar datos de Finanzas
PROCESO FINALIZADO : Datos enviados a MongoDB.


In [21]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [22]:
1 # Esto nos dice que columnas entendio Spark que habia en Mongo
df.printSchema()
# Esto nos muestra las primeras 5 filas para confirmar que hay datos reales
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+--------------------+--------------------+-----+
|                 _id|categoria|      fecha_captura|               grupo|                item|valor|
+--------------------+---------+-------------------+--------------------+--------------------+-----+
|69dd5e1aa52d4580f...| Finanzas|2026-04-13 21:20:26|G3_FinanzasyMerca...|  It’s Only Himalaya|45.17|
|69dd5e1aa52d4580f...| Finanzas|2026-04-13 21:20:26|G3_FinanzasyMerca...|Full Moon over No...|49.43|
|69dd62d5eda5cb3f4...| Finanzas|2026-04-13 21:40:37|G3_FinanzasyMerca...|  It’s Only Himalaya|45.17|
|69dd62d5eda5cb3f4...| Finanzas|2026-04-13 21:40:37|G3_FinanzasyMerca...|Full Moon over No...|49.43|
+--------------------+---------+-------------------+----

In [23]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 4
Registros validos: 4


In [24]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It’s Only Himalaya|45.17|
|Full Moon over No...|49.43|
|  It’s Only Himalaya|45.17|
|Full Moon over No...|49.43|
+--------------------+-----+



In [25]:
df.filter(df["valor"] > 50).show()

+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+



In [26]:
df.groupBy("grupo").count().show()

+--------------------+-----+
|               grupo|count|
+--------------------+-----+
|G3_FinanzasyMerca...|    4|
+--------------------+-----+



In [27]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+---------------+-------------+-------------+
|categoria|cantidad_libros|precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+---------------+-------------+-------------+
| Finanzas|              4|           47.3|        45.17|        49.43|
+---------+---------------+---------------+-------------+-------------+



In [28]:
fila = df.orderBy(F.desc("valor")).select(
    "item", "categoria", "fecha_captura", "valor"
).first()

print("Producto más caro:", fila["item"])
print("Categoría:", fila["categoria"])
print("Fecha y hora de captura:", fila["fecha_captura"])
print("Valor:", fila["valor"])

Producto más caro: Full Moon over Noahs Ark
Categoría: Finanzas
Fecha y hora de captura: 2026-04-13 21:20:26
Valor: 49.43


In [33]:
from pyspark.sql import functions as F

# CONSULTA DE SALIDA: Resumen de integridad por Categoría
ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupby("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()


--- TICKET DE SALIDA: G3_FinanzasyMercadoortega ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
| Finanzas|           6|        47.3|  2026-04-13 21:40:56|
+---------+------------+------------+---------------------+



In [31]:
print(df.columns)


['_id', 'categoria', 'fecha_captura', 'grupo', 'item', 'valor']
